**BPE tokenizer from scratch**

Build a working Byte-Pair Encoding tokenizer to understand how GPT-style tokenization works

In [ ]:
# ============================================================
# BPE TOKENIZER FROM SCRATCH
# Runs in browser! No external dependencies needed.
# ============================================================

import re
from collections import Counter

class SimpleBPETokenizer:
    """
    A minimal BPE tokenizer that shows exactly how GPT-style
    tokenization works. Real tokenizers use this same algorithm!
    """

    def __init__(self):
        self.vocab = {}          # token -> id
        self.inverse_vocab = {}  # id -> token
        self.merges = []         # list of (pair_to_merge, new_token)

    def get_pairs(self, word):
        """Get all adjacent pairs in a word."""
        pairs = Counter()
        chars = list(word)
        for i in range(len(chars) - 1):
            pairs[(chars[i], chars[i + 1])] += 1
        return pairs

    def train(self, text, vocab_size=50):
        """
        Train BPE: repeatedly merge most common pairs.
        This is exactly what GPT's tokenizer training does!
        """
        # Start with character-level tokens
        words = text.split()
        # Add end-of-word marker (like GPT uses spaces)
        word_freqs = Counter([''.join(list(w) + ['</w>']) for w in words])

        # Initialize vocab with all characters
        all_chars = set(''.join(word_freqs.keys()))
        self.vocab = {ch: i for i, ch in enumerate(sorted(all_chars))}
        self.inverse_vocab = {i: ch for ch, i in self.vocab.items()}

        print(f"Starting vocab: {len(self.vocab)} characters")
        print(f"Training to vocab size: {vocab_size}")
        print("-" * 40)

        # BPE: merge most frequent pairs
        while len(self.vocab) < vocab_size:
            # Count all pairs across all words
            pair_counts = Counter()
            for word, freq in word_freqs.items():
                pairs = self.get_pairs(word)
                for pair, count in pairs.items():
                    pair_counts[pair] += count * freq

            if not pair_counts:
                break

            # Find and merge most frequent pair
            best_pair = pair_counts.most_common(1)[0][0]
            new_token = best_pair[0] + best_pair[1]

            print(f"Merge #{len(self.merges)+1}: "
                  f"'{best_pair[0]}' + '{best_pair[1]}' -> '{new_token}'")

            # Add to vocab
            new_id = len(self.vocab)
            self.vocab[new_token] = new_id
            self.inverse_vocab[new_id] = new_token
            self.merges.append((best_pair, new_token))

            # Apply merge to all words
            new_word_freqs = {}
            for word, freq in word_freqs.items():
                new_word = word.replace(
                    best_pair[0] + best_pair[1], new_token
                )
                new_word_freqs[new_word] = freq
            word_freqs = new_word_freqs

        print("-" * 40)
        print(f"Final vocab size: {len(self.vocab)}")

    def tokenize(self, text):
        """Tokenize text using learned merges."""
        words = text.split()
        all_tokens = []

        for word in words:
            # Start with characters
            tokens = list(word) + ['</w>']

            # Apply merges in order learned
            for (pair, new_token) in self.merges:
                i = 0
                while i < len(tokens) - 1:
                    if (tokens[i] == pair[0]
                            and tokens[i + 1] == pair[1]):
                        tokens = (tokens[:i] + [new_token]
                                  + tokens[i + 2:])
                    else:
                        i += 1

            all_tokens.extend(tokens)

        return all_tokens

    def encode(self, text):
        """Convert text to token IDs."""
        tokens = self.tokenize(text)
        return [self.vocab.get(t, 0) for t in tokens]

    def decode(self, ids):
        """Convert token IDs back to text."""
        tokens = [self.inverse_vocab.get(i, '?') for i in ids]
        text = ''.join(tokens).replace('</w>', ' ')
        return text.strip()


# ============================================================
# DEMO: Train and use the tokenizer
# ============================================================

corpus = """
the cat sat on the mat
the dog sat on the log
the cat and the dog played
"""

print("=" * 50)
print("BPE TOKENIZER TRAINING")
print("=" * 50)

tokenizer = SimpleBPETokenizer()
tokenizer.train(corpus, vocab_size=30)

# Test tokenization
print("\n" + "=" * 50)
print("TOKENIZATION EXAMPLES")
print("=" * 50)

for text in ["the cat", "the dog sat", "cat and dog"]:
    tokens = tokenizer.tokenize(text)
    ids = tokenizer.encode(text)
    print(f"\nText: '{text}'")
    print(f"  Tokens: {tokens}")
    print(f"  IDs:    {ids}")
    print(f"  Decoded: '{tokenizer.decode(ids)}'")

print("\n" + "=" * 50)
print("KEY INSIGHT: BPE builds vocabulary by merging")
print("the most common adjacent pairs - exactly how")
print("GPT-4o was trained, just on much more text!")
print("=" * 50)